# 00 — Taobao dataset study (before ingestion)

**Purpose:** understand each CSV and each column *before* writing any pipeline code.
This notebook is for studying only. The real pipeline lives in `src/data/`.

**Rule used here:** the two small tables are read in full. `raw_sample` (1.1 GB) is read in
full through DuckDB. `behavior_log` (23.7 GB) is only **sampled** (first rows), because a
full scan is the job of `src/data/dq_report.py`.

In [1]:
from pathlib import Path
import sys, json, time
import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw" / "taobao"
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
con = duckdb.connect()
con.execute("SET memory_limit='16GB'")
con.execute("SET threads=8")
# Taobao timestamps are Beijing time: 1494000000 is exactly 2017-05-06 00:00 at UTC+8.
# Without this, DuckDB uses the machine's own timezone and the per-day buckets are shifted.
con.execute("SET TimeZone='Asia/Shanghai'")
print("project root:", ROOT)

project root: /data/home/mitanshub/Ad_ranking_Bidding_Stepwise


## 1. The four files

In [2]:
rows = []
for f in ["raw_sample", "behavior_log", "ad_feature", "user_profile"]:
    p = RAW / f"{f}.csv"
    with open(p) as fh:
        header = fh.readline().rstrip("\n")
    rows.append({"file": f"{f}.csv", "size_GB": round(p.stat().st_size / 1e9, 3), "header": header})
pd.DataFrame(rows)

,file,size_GB,header
0,raw_sample.csv,1.088,"user,time_stamp,adgroup_id,pid,nonclk,clk"
1,behavior_log.csv,23.729,"user,time_stamp,btag,cate,brand"
2,ad_feature.csv,0.031,"adgroup_id,cate_id,campaign_id,customer,brand,..."
3,user_profile.csv,0.024,"userid,cms_segid,cms_group_id,final_gender_cod..."


**How to read this**
- `behavior_log` is 23.7 GB of the 24.9 GB total (about 95%). Almost every expensive step in this project is about this one file.
- Each file is a different "view" of the same shop: impressions (`raw_sample`), past behaviour (`behavior_log`), ad attributes (`ad_feature`), user attributes (`user_profile`).
- Only `user_profile` names its key `userid`; the other tables use `user`. Its last header also has a trailing space that the display above hides.

## 2. Column dictionary

Meanings below are from the public Tianchi *Ad Display/Click Data on Taobao.com* description,
written from memory. Anything marked **(verify)** should be checked against the data in the
sections that follow, not trusted.

In [3]:
dictionary = [
 # table, column, documented meaning
 ("raw_sample","user","anonymised user ID (join key to behavior_log and user_profile.userid)"),
 ("raw_sample","time_stamp","impression time, unix seconds (Beijing time, UTC+8, confirmed from the data)"),
 ("raw_sample","adgroup_id","ad unit ID (join key to ad_feature)"),
 ("raw_sample","pid","ad slot / resource position ID"),
 ("raw_sample","nonclk","1 = impression was NOT clicked"),
 ("raw_sample","clk","1 = impression WAS clicked  <- the label"),
 ("behavior_log","user","anonymised user ID"),
 ("behavior_log","time_stamp","when the behaviour happened, unix seconds"),
 ("behavior_log","btag","behaviour type: pv (browse), cart, fav, buy"),
 ("behavior_log","cate","category ID of the item acted on"),
 ("behavior_log","brand","brand ID of the item acted on"),
 ("ad_feature","adgroup_id","ad unit ID (primary key)"),
 ("ad_feature","cate_id","category ID of the advertised item"),
 ("ad_feature","campaign_id","campaign ID"),
 ("ad_feature","customer","advertiser ID"),
 ("ad_feature","brand","brand ID (may be missing)"),
 ("ad_feature","price","item price (currency/unit: verify)"),
 ("user_profile","userid","user ID (named `userid`, not `user`)"),
 ("user_profile","cms_segid","micro-group ID"),
 ("user_profile","cms_group_id","group ID"),
 ("user_profile","final_gender_code","1 = male, 2 = female"),
 ("user_profile","age_level","age bucket (levels: verify)"),
 ("user_profile","pvalue_level","consumption level: 1 low, 2 mid, 3 high (missing is common)"),
 ("user_profile","shopping_level","shopping depth: 1 light, 2 moderate, 3 heavy"),
 ("user_profile","occupation","1 = college student, 0 = other"),
 ("user_profile","new_user_class_level ","city level (NOTE the trailing space in the real header)"),
]
pd.DataFrame(dictionary, columns=["table","column","documented meaning"])

,table,column,documented meaning
0,raw_sample,user,anonymised user ID (join key to behavior_log a...
1,raw_sample,time_stamp,"impression time, unix seconds (Beijing time, U..."
2,raw_sample,adgroup_id,ad unit ID (join key to ad_feature)
3,raw_sample,pid,ad slot / resource position ID
4,raw_sample,nonclk,1 = impression was NOT clicked
5,raw_sample,clk,1 = impression WAS clicked <- the label
6,behavior_log,user,anonymised user ID
7,behavior_log,time_stamp,"when the behaviour happened, unix seconds"
8,behavior_log,btag,"behaviour type: pv (browse), cart, fav, buy"
9,behavior_log,cate,category ID of the item acted on


## 3. `raw_sample` — ad impressions (the main table, one row per impression)

In [4]:
con.execute(f"CREATE OR REPLACE VIEW raw_sample AS SELECT * FROM read_csv('{RAW/'raw_sample.csv'}', header=true)")
display(con.execute("DESCRIBE raw_sample").df())
display(con.execute("SELECT * FROM raw_sample LIMIT 5").df())

,column_name,column_type,null,key,default,extra
0,user,BIGINT,YES,None,None,None
1,time_stamp,BIGINT,YES,None,None,None
2,adgroup_id,BIGINT,YES,None,None,None
3,pid,VARCHAR,YES,None,None,None
4,nonclk,BIGINT,YES,None,None,None
5,clk,BIGINT,YES,None,None,None


,user,time_stamp,adgroup_id,pid,nonclk,clk
0,581738,1494137644,1,430548_1007,1,0
1,449818,1494638778,3,430548_1007,1,0
2,914836,1494650879,4,430548_1007,1,0
3,914836,1494651029,5,430548_1007,1,0
4,399907,1494302958,8,430548_1007,1,0


**How to read this**
- One row = one ad shown to one user at one moment. There is no row for "not shown", so we only ever see what the old system chose to show.
- `time_stamp` is unix seconds. `pid` is text (an ad slot). `nonclk` and `clk` are 0/1 flags.

In [5]:
t0 = time.time()
summary = con.execute("""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT "user") AS users,
           COUNT(DISTINCT adgroup_id) AS adgroups,
           COUNT(DISTINCT pid) AS pids,
           AVG(clk) AS click_rate,
           MIN(time_stamp) AS ts_min, MAX(time_stamp) AS ts_max,
           to_timestamp(MIN(time_stamp)) AS first_impression_cst,
           to_timestamp(MAX(time_stamp)) AS last_impression_cst
    FROM raw_sample""").df()
print(f"{time.time()-t0:.0f}s")
summary.T

1s


,0
rows,26557961
users,1141729
adgroups,846811
pids,2
click_rate,0.051437
ts_min,1494000000
ts_max,1494691186
first_impression_cst,2017-05-06 00:00:00+08:00
last_impression_cst,2017-05-13 23:59:46+08:00


**How to read this**
- 26.6M impressions from 1.14M users is about 23 impressions per user, and 846,811 ad units is about 31 impressions per ad unit.
- **Click rate 5.14%** (about 1 in 19). A model that always says "no click" would be 94.9% accurate, so accuracy is useless here. We use AUC, GAUC, LogLoss and calibration instead.
- The time range is exactly 8 days (2017-05-06 00:00 to 2017-05-13 23:59, Beijing time). The plan is to train on days 1-7 and test on day 8.
- Only 2 distinct `pid` values, so slot is a very low-cardinality feature.

In [6]:
print("clk vs nonclk — are they exact complements?")
display(con.execute("SELECT clk, nonclk, COUNT(*) AS n FROM raw_sample GROUP BY ALL ORDER BY 1,2").df())
print("pid (ad slots)")
display(con.execute("SELECT pid, COUNT(*) AS n, AVG(clk) AS ctr FROM raw_sample GROUP BY pid").df())
print("impressions per day (Beijing time, UTC+8)")
display(con.execute("""SELECT CAST(to_timestamp(time_stamp) AS DATE) AS day, COUNT(*) AS n, AVG(clk) AS ctr
                       FROM raw_sample GROUP BY day ORDER BY day""").df())

clk vs nonclk — are they exact complements?


,clk,nonclk,n
0,0,1,25191905
1,1,0,1366056


pid (ad slots)


,pid,n,ctr
0,430539_1007,10085063,0.053578
1,430548_1007,16472898,0.050126


impressions per day (Beijing time, UTC+8)


,day,n,ctr
0,2017-05-06,3270348,0.052084
1,2017-05-07,3430002,0.051248
2,2017-05-08,3354523,0.051988
3,2017-05-09,3248016,0.052218
4,2017-05-10,3333752,0.051651
5,2017-05-11,3378604,0.052633
6,2017-05-12,3234051,0.049299
7,2017-05-13,3308665,0.050319


**How to read this**
- `clk` and `nonclk` are exact opposites (every row is either 0/1 or 1/0). `nonclk` is just the label written backwards. It must **never** be used as a feature, or the model would see the answer.
- The two slots have similar click rates (5.0% and 5.4%), so slot alone says little.
- Daily volume is steady (3.2-3.4M impressions) and the daily click rate stays within 4.9-5.3%, so there is no big drift across days. The last day (2017-05-13, about 3.31M rows) is our test day.

In [7]:
print("impressions per user: distribution")
display(con.execute("""
    WITH c AS (SELECT "user", COUNT(*) n, SUM(clk) clicks FROM raw_sample GROUP BY "user")
    SELECT quantile_cont(n,0.5) p50, quantile_cont(n,0.9) p90, quantile_cont(n,0.99) p99, MAX(n) max_n,
           AVG((clicks=0)::INT) AS share_users_zero_clicks,
           AVG((clicks=n)::INT) AS share_users_all_clicks
    FROM c""").df())

impressions per user: distribution


,p50,p90,p99,max_n,share_users_zero_clicks,share_users_all_clicks
0,7.0,48.0,286.0,3756,0.569384,0.003813


**How to read this**
- Very heavy-tailed: the median user has 7 impressions, but the top 1% have 286 or more, and the busiest has 3,756.
- 56.9% of users never clicked and 0.4% always clicked. So **only about 43% of users have both a click and a non-click**, and only those can be scored by GAUC. Users with few impressions give noisy per-user scores.

## 4. `ad_feature` — one row per ad unit

In [8]:
ad = pd.read_csv(RAW / "ad_feature.csv")
print(ad.shape)
display(ad.head())
display(ad.dtypes.to_frame("dtype").join(ad.isna().sum().to_frame("nulls")).join(ad.nunique().to_frame("distinct")))
display(ad.describe(percentiles=[.5, .9, .99, .999]).T)

(846811, 6)


,adgroup_id,cate_id,campaign_id,customer,brand,price
0,63133,6406,83237,1,95471.0,170.00
1,313401,6406,83237,1,87331.0,199.00
2,248909,392,83237,1,32233.0,38.00
3,208458,392,83237,1,174374.0,139.00
4,110847,7211,135256,2,145952.0,32.99


,dtype,nulls,distinct
adgroup_id,int64,0,846811
cate_id,int64,0,6769
campaign_id,int64,0,423436
customer,int64,0,255875
brand,float64,246330,99814
price,float64,0,14861


,count,mean,std,min,50%,90%,99%,99.9%,max
adgroup_id,846811.0,423406.000000,244453.423739,1.00,423406.0,762130.0,838342.9,845964.19,846811.0
cate_id,846811.0,5868.593464,2705.171203,1.00,6183.0,9713.0,12205.0,12845.00,12960.0
campaign_id,846811.0,206552.604280,125192.340908,1.00,200780.0,388315.0,420016.0,423058.00,423436.0
customer,846811.0,113180.406006,73435.834950,1.00,107278.0,221871.0,252090.0,255443.00,255875.0
brand,600481.0,229254.422596,132288.849633,2.00,234423.0,412090.0,455132.0,460833.00,461497.0
price,846811.0,1838.867099,310887.697300,0.01,139.0,970.0,8720.0,51006.84,99999999.0


**How to read this**
- 846,811 rows with a unique `adgroup_id`: one row per ad unit.
- The ID columns have huge cardinality (6,769 categories, 423,436 campaigns, 255,875 advertisers, 99,814 brands). These are identifiers, not quantities, so they need embeddings or hashing, not one-hot columns.
- **`brand` has 246,330 missing values (29%).**
- **`price` is extremely skewed:** median 139, 99th percentile 8,720, 99.9th percentile about 51,007, but the maximum is 99,999,999. The mean (1,839) is dragged up by such values. Expect to cap and log-transform it.

In [9]:
# does the literal text "NULL" occur anywhere? (pandas treats it as NaN by default, so check raw text)
raw_ad = pd.read_csv(RAW / "ad_feature.csv", dtype=str, keep_default_na=False)
print("literal 'NULL' per column:"); print((raw_ad == "NULL").sum().to_dict())
print("empty strings per column:"); print((raw_ad == "").sum().to_dict())
print("adgroup_id unique?", ad.adgroup_id.is_unique)

literal 'NULL' per column:
{'adgroup_id': 0, 'cate_id': 0, 'campaign_id': 0, 'customer': 0, 'brand': 246330, 'price': 0}
empty strings per column:
{'adgroup_id': 0, 'cate_id': 0, 'campaign_id': 0, 'customer': 0, 'brand': 0, 'price': 0}
adgroup_id unique? True


**How to read this**
- The missing brands are written as the literal text `NULL`, not as empty cells. Pandas hides this by turning the text into NaN. A plain DuckDB read would keep it as text and turn the whole `brand` column into strings.
- That is why the ingest script reads with `nullstr=['NULL','']` and an explicit integer type.

## 5. `user_profile` — one row per user

In [10]:
up = pd.read_csv(RAW / "user_profile.csv")
print(up.shape)
print("column names (repr shows hidden whitespace):", [c for c in up.columns])
display(up.head())
display(up.isna().sum().to_frame("nulls").assign(null_rate=lambda d: (d.nulls / len(up)).round(4)))

(1061768, 9)
column names (repr shows hidden whitespace): ['userid', 'cms_segid', 'cms_group_id', 'final_gender_code', 'age_level', 'pvalue_level', 'shopping_level', 'occupation', 'new_user_class_level ']


,userid,cms_segid,cms_group_id,final_gender_code,age_level,pvalue_level,shopping_level,occupation,new_user_class_level
0,234,0,5,2,5,NaN,3,0,3.0
1,523,5,2,2,2,1.0,3,1,2.0
2,612,0,8,1,2,2.0,3,0,NaN
3,1670,0,4,2,4,NaN,1,0,NaN
4,2545,0,10,1,4,NaN,3,0,NaN


,nulls,null_rate
userid,0,0.0000
cms_segid,0,0.0000
cms_group_id,0,0.0000
final_gender_code,0,0.0000
age_level,0,0.0000
pvalue_level,575917,0.5424
shopping_level,0,0.0000
occupation,0,0.0000
new_user_class_level,344920,0.3249


**How to read this**
- 1,061,768 users, one row each (`userid` is unique).
- Two columns have a lot of missing values: `pvalue_level` (54%) and `new_user_class_level` (32%). The other columns are complete.
- Missingness is probably not random (for example, users with little history). Treat "missing" as its own category instead of dropping rows or filling with an average.

In [11]:
for c in ["final_gender_code", "age_level", "pvalue_level", "shopping_level", "occupation", up.columns[-1]]:
    print(f"--- {c!r}")
    print(up[c].value_counts(dropna=False).sort_index().to_dict())
print("userid unique?", up.userid.is_unique)

--- 'final_gender_code'
{1: 377517, 2: 684251}
--- 'age_level'
{0: 501, 1: 65568, 2: 189617, 3: 307340, 4: 261751, 5: 214462, 6: 22529}
--- 'pvalue_level'
{1.0: 154436, 2.0: 293656, 3.0: 37759, nan: 575917}
--- 'shopping_level'
{1: 74612, 2: 151411, 3: 835745}
--- 'occupation'
{0: 1002064, 1: 59704}
--- 'new_user_class_level '
{1.0: 80548, 2.0: 324420, 3.0: 173047, 4.0: 138833, nan: 344920}
userid unique? True


**How to read this**
- All of these are small integer **codes** (levels), not measurements. Treat them as categories (the ordered ones can also be used as numbers).
- Gender: 64% are code 2 and 36% are code 1. Shopping level 3 covers about 79% of users, so that column carries little signal on its own. Occupation code 1 is only 5.6%.
- `age_level` has 501 users with level 0, which does not belong with levels 1-6. Treat as unknown.
- Every value count is listed with the missing ones, so the `pvalue_level` gap (575,917 users) is visible next to the real levels.

## 6. `behavior_log` — the second surface (723M events, **sampled only**)

The file is probably ordered in some way (by user or time), so the first rows are **not** a
random sample. Use this only to understand the columns, never to estimate rates.

In [12]:
SAMPLE = 5_000_000
con.execute(f"""CREATE OR REPLACE TABLE bl_sample AS
    SELECT * FROM read_csv('{RAW/'behavior_log.csv'}', header=true,
                           columns={{'user':'BIGINT','time_stamp':'BIGINT','btag':'VARCHAR','cate':'VARCHAR','brand':'VARCHAR'}})
    LIMIT {SAMPLE}""")
display(con.execute("SELECT * FROM bl_sample LIMIT 8").df())
print("btag values:")
display(con.execute("SELECT btag, COUNT(*) n FROM bl_sample GROUP BY btag ORDER BY n DESC").df())

,user,time_stamp,btag,cate,brand
0,558157,1493741625,pv,6250,91286
1,558157,1493741626,pv,6250,91286
2,558157,1493741627,pv,6250,91286
3,728690,1493776998,pv,11800,62353
4,332634,1493809895,pv,1101,365477
5,857237,1493816945,pv,1043,110616
6,619381,1493774638,pv,385,428950
7,467042,1493772641,pv,8237,301299


btag values:


,btag,n
0,pv,4757711
1,cart,110009
2,buy,67720
3,fav,64560


**How to read this**
- One row = one behaviour event by one user. The first three rows are the same user, one second apart, same category and brand: repeated page views of the same item.
- Event mix in this sample: browsing (`pv`) is about 95%, cart 2.2%, buy 1.4%, favourite 1.3%. Purchases are rare but probably the strongest signal of interest.
- Caution: these are the **first** rows of the file, not a random sample. Use them to understand columns, not to estimate rates.

In [13]:
# read cate/brand as text on purpose, to see whether non-numeric junk such as 'NULL' appears
display(con.execute("""
    SELECT SUM((brand = 'NULL')::INT) AS brand_NULL_text,
           SUM((brand IS NULL)::INT)  AS brand_empty,
           SUM((cate  = 'NULL')::INT) AS cate_NULL_text,
           SUM((cate  IS NULL)::INT)  AS cate_empty,
           SUM((TRY_CAST(cate AS BIGINT)  IS NULL AND cate  IS NOT NULL)::INT) AS cate_not_numeric,
           SUM((TRY_CAST(brand AS BIGINT) IS NULL AND brand IS NOT NULL)::INT) AS brand_not_numeric
    FROM bl_sample""").df())
display(con.execute("""
    SELECT COUNT(DISTINCT "user") users_in_sample,
           to_timestamp(MIN(time_stamp)) AS first_cst, to_timestamp(MAX(time_stamp)) AS last_cst
    FROM bl_sample""").df())
print("exact duplicate rows within the sample:",
      con.execute("SELECT COUNT(*) - COUNT(DISTINCT (\"user\", time_stamp, btag, cate, brand)) FROM bl_sample").fetchone()[0])

,brand_NULL_text,brand_empty,cate_NULL_text,cate_empty,cate_not_numeric,brand_not_numeric
0,0.0,0.0,0.0,0.0,0.0,0.0


,users_in_sample,first_cst,last_cst
0,508423,1918-07-27 16:05:29+08:00,2032-11-26 14:16:14+08:00


exact duplicate rows within the sample: 349113


**How to read this**
- In this sample `cate` and `brand` are always plain numbers, with no `NULL` text. The text problem is in `ad_feature`, not here.
- The earliest timestamp in the sample is in 1918 and the latest in 2032. Both are far outside 2017, so `behavior_log` contains **corrupt timestamps** that must be filtered. This matters a lot: our rule "only use behaviour strictly before the impression" depends on correct timestamps.
- 349,113 of 5M rows (7.0%) are exact duplicates. They may be genuine repeated views in the same second or logging artefacts. We decide after the full-file numbers from `dq_report.py`.

## 7. How the tables join — key coverage checks that are cheap

In [14]:
con.register("ad_df", ad)
con.register("up_df", up)
display(con.execute("""
    SELECT COUNT(DISTINCT r.adgroup_id) AS adgroups_in_samples,
           COUNT(DISTINCT a.adgroup_id) AS adgroups_found_in_ad_feature
    FROM raw_sample r LEFT JOIN ad_df a USING (adgroup_id)""").df())
display(con.execute("""
    SELECT COUNT(DISTINCT r."user") AS users_in_raw_sample,
           COUNT(DISTINCT u.userid) AS users_found_in_user_profile
    FROM raw_sample r LEFT JOIN up_df u ON r."user" = u.userid""").df())

,adgroups_in_samples,adgroups_found_in_ad_feature
0,846811,846811


,users_in_raw_sample,users_found_in_user_profile
0,1141729,1061768


**How to read this**
- Every ad unit in `raw_sample` has a row in `ad_feature`, so the ad join is lossless.
- 79,961 users (7%) in `raw_sample` have no `user_profile` row. Dropping them would remove their impressions, so keep them and mark the profile features as unknown.
- Whether a user has any behaviour history cannot be checked cheaply; `dq_report.py` measures that over the full 723M rows.

```
raw_sample.user        ──┬── user_profile.userid      (user features)
raw_sample.adgroup_id  ──── ad_feature.adgroup_id     (ad features)
raw_sample.user (+ time_stamp, strictly earlier) ──── behavior_log.user   (past behaviour; the cross-surface signal)
```
Joining `behavior_log` is the expensive step. Coverage there is measured by `src/data/dq_report.py`.